In [ ]:
# binned sequential MC path finding in parallel

# define a function that runs binned monte carlo path finding
def binned_mc_path_finding(mc_run):

    if start_from_endstates == True:
        # starting structure is one of the end states
        starting_structure = ensemble_df.sort_values(by=['rmsd_to_outward']).head(1).index.values[0]
        #starting_structure = ensemble_df.sort_values(by=['rmsd_to_inward']).head(1).index.values[0]
    else:
        # starting structure is a random structure from the first bin
        starting_structure = np.random.choice(ensemble_df[ensemble_df['bin'] == 0].index.values)
    
    starting_bin = 0

    temperature = mc_temp
    wf_sasa = mc_wf_sasa
    n_bins = mc_n_bins

    mcpath = []    
    mcpath.append(starting_structure)
    current_path_length = len(mcpath)

    # while the current structure is not in bin n_bins
    current_structure = starting_structure
    next_bin = starting_bin+1
    while next_bin != n_bins:

        # the energy proxy is the sum of the square rmsd between each structure in the path
        energy = 0
        for i in range(len(mcpath) - 1):
            energy += np.sqrt ( 1/current_path_length * (rmsd_matrix[np.where(ensemble_df.index.values == mcpath[i])[0][0], np.where(ensemble_df.index.values == mcpath[i+1])[0][0]]**2) )
            energy += sasa_matrix[np.where(ensemble_df.index.values == mcpath[i])[0][0], np.where(ensemble_df.index.values == mcpath[i+1])[0][0]]*wf_sasa

        # propose a move by selecting a random non-self structure from the pool
        structure_pool = ensemble_df[ensemble_df['bin'] == next_bin].index.values
        proposed_structure = np.random.choice(structure_pool[structure_pool != current_structure])

        # proposed path
        new_mcpath = mcpath.copy()
        new_mcpath.append(proposed_structure)

        # calcualte energy of proposed path
        new_energy = 0
        for i in range(len(new_mcpath) - 1):
            new_energy += np.sqrt ( 1/current_path_length * (rmsd_matrix[np.where(ensemble_df.index.values == new_mcpath[i])[0][0], np.where(ensemble_df.index.values == new_mcpath[i+1])[0][0]]**2) )
            new_energy += sasa_matrix[np.where(ensemble_df.index.values == new_mcpath[i])[0][0], np.where(ensemble_df.index.values == new_mcpath[i+1])[0][0]]*wf_sasa

        # calculate the difference between the two energies
        delta_energy = new_energy - energy

        # metropolis criterion
        if delta_energy < 0:
            current_structure = proposed_structure
            mcpath.append(current_structure)
            # if the proposed structure is in the next bin, increment next_bin
            if ensemble_df.loc[proposed_structure]['bin'] == next_bin:
                next_bin += 1
        # if higher, accept with probability e^(-difference/temp)
        else:
            if np.random.rand() < np.exp(-delta_energy / temperature):
                current_structure = proposed_structure
                mcpath.append(current_structure)
                # if the proposed structure is in the next bin, increment next_bin
                if ensemble_df.loc[proposed_structure]['bin'] == next_bin:
                    next_bin += 1
            else:
                pass

    # calculate the energy of the final path
    energy = 0
    for i in range(len(mcpath) - 1):
        energy += np.sqrt ( 1/current_path_length * (rmsd_matrix[np.where(ensemble_df.index.values == mcpath[i])[0][0], np.where(ensemble_df.index.values == mcpath[i+1])[0][0]]**2) )
        energy += sasa_matrix[np.where(ensemble_df.index.values == mcpath[i])[0][0], np.where(ensemble_df.index.values == mcpath[i+1])[0][0]]*wf_sasa

    #  replace mcpath entries (which are indices of hte dataframe) with the structure names from the 'structure' column
    #mcpath = ensemble_df.loc[mcpath]['structure'].values
    return [energy, mcpath, ensemble_df.loc[mcpath]['structure'].values]

mc_runs = []

# run once
mcruns_initial = binned_mc_path_finding(1)

# get the structure indices
print(mcruns_initial[1])
# drop the frist and last structures
mcruns_initial = mcruns_initial[1][1:-1]


#
## run mc_n_bins runs in parallel
#with Pool(processes=num_processes) as pool:
#    mc_runs = list(tqdm(pool.imap(binned_mc_path_finding, range(mc_n_runs)), total=mc_n_runs))
#
## turn this mc_runs list object into a pandas dataframe
#mc_runs_df = pd.DataFrame(mc_runs, columns=['energy', 'path', 'path structures'])
#mc_runs_df = mc_runs_df.sort_values(by=['energy'])
#
## write out the best run to a multistate pdb file and append timestamp to filename
#with mda.Writer(base_directory + 'best_run.pdb', u.atoms.n_atoms) as W:
#    # get the structures of the lowest energy path
#    for structure in mc_runs_df.head(1)['path structures'].values[0]:
#        # if structure is the reference, dont renumber residues
#        if structure == 'ref_outward.pdb' or structure == 'ref_inward.pdb':
#            # use a non-reference structure to make the universe
#            u = mda.Universe(structure_directory + structure, 
#                             structure_directory + structure) # make universe
#            u.atoms.residues.resids += resid_offset 
#            u.atoms.segments.segids = 'A'
#            u.atoms.chainIDs = 'A'
#            W.write(u.select_atoms('protein'))
#        else:
#            u = mda.Universe(structure_directory + structure, 
#                             structure_directory + structure) # make universe
#            u.atoms.residues.resids += resid_offset
#            W.write(u.select_atoms('protein'))
        

In [ ]:
# "intertial" sasa relaxation

n_max_steps = 100000
max_num_rejections = 1000
temperature = 1

def calc_sasa_energy(new_structure, current_structure):

    wf_rmsd = 5
    # get the difference in rmsd from the rmsd_matrix
    energy_rmsd = 0
    energy_rmsd = wf_rmsd * np.sqrt(rmsd_matrix[np.where(ensemble_df.index.values == new_structure)[0][0], np.where(ensemble_df.index.values == current_structure)[0][0]]**2)

    wf_collective_variable = 0
    # get the difference in collective variable from the cv_matrix
    energy_collective_variable = 0
    energy_collective_variable = wf_collective_variable * np.sqrt(cv_matrix[np.where(ensemble_df.index.values == new_structure)[0][0], np.where(ensemble_df.index.values == current_structure)[0][0]]**2)

    wf_sasa = 1
    # for miminising sasa
    energy_sasa = 0
    energy_sasa = -1 * wf_sasa * sasa_matrix[np.where(ensemble_df.index.values == new_structure)[0][0], np.where(ensemble_df.index.values == current_structure)[0][0]]

    # final term is sum
    energy = energy_rmsd + energy_sasa + energy_collective_variable

    # final energy
    return energy

# function to rnadomly wawlk through the ensemble and return a path
def mc_sasa_minimisation(seed):
    
    # random seed
    np.random.seed(seed)

    mcpath = []    
    num_rejections = 0

    # starting structure is a random structure from the ensemble
    #starting_structure = ensemble_df.sample(1).index.values[0]
    
    # starting structure is the outward facing reference structure (index)
    starting_structure = ensemble_df.loc[ensemble_df['structure'] == 'ref_outward.pdb'].index.values[0]

    # structure pool consists of the names of the structures in the ensemble dataframe
    structure_pool = ensemble_df.index.values

    # add the starting structure to the path
    mcpath.append(starting_structure)

    # remove the starting structure from the pool
    structure_pool = np.delete(structure_pool, np.where(structure_pool == starting_structure)[0][0])

    current_structure = starting_structure

    # loop indefinitely (until n_max_steps is reached)
    #for i in range(n_max_steps):

    #loop until num_rejections > max_num_rejections
    while num_rejections < max_num_rejections:
            
        ## calculate energy of current path
       # energy = calc_sasa_energy(mcpath)

        # select a random structure from the pool (and then remove it from the pool later if it is accepted)
        random_structure_in_pool = np.random.choice(structure_pool)

        # calcualte energy of proposed path
        delta_energy = calc_sasa_energy(current_structure, random_structure_in_pool)

        # metropolis criterion
    
        # if lower, accept the new path
        if delta_energy < 0:

            current_structure = random_structure_in_pool
            mcpath.append(current_structure)

            num_rejections = 0
            #print(delta_energy)

        else:

            # if higher, accept with probability e^(-difference/temp)
            if np.random.rand() < np.exp(-delta_energy / temperature):

                current_structure = random_structure_in_pool
                mcpath.append(current_structure)

                num_rejections = 0
                #print(delta_energy)

            else:

                # increment a counter for the number of times the path is rejected
                num_rejections += 1
                mcpath.append(current_structure)
                pass

    return mcpath

# call the function
sasa_min_path = mc_sasa_minimisation(0)

# plot the path on the 2D plot
plt.scatter(ensemble_df[plot_variable_1], ensemble_df[plot_variable_2], c=ensemble_df[plot_variable_3], cmap='coolwarm')
plt.xlabel(plot_variable_1)
plt.ylabel(plot_variable_2)
plt.colorbar(label=plot_variable_3)
plt.annotate('outward facing', (ensemble_df.loc[ensemble_df['structure'] == 'ref_outward.pdb'][plot_variable_1], ensemble_df.loc[ensemble_df['structure'] == 'ref_outward.pdb'][plot_variable_2]))
plt.annotate('inward facing', (ensemble_df.loc[ensemble_df['structure'] == 'ref_inward.pdb'][plot_variable_1], ensemble_df.loc[ensemble_df['structure'] == 'ref_inward.pdb'][plot_variable_2]))
plt.plot(ensemble_df[plot_variable_1].loc[sasa_min_path], ensemble_df[plot_variable_2].loc[sasa_min_path], c='black')

# annotate the most common structure in the path
plt.annotate('most common structure', (ensemble_df[plot_variable_1].loc[sasa_min_path].value_counts().index[0], ensemble_df[plot_variable_2].loc[sasa_min_path].value_counts().index[0]))
# get the name of this structure 
most_common_structure = ensemble_df[plot_variable_1].loc[sasa_min_path].value_counts().index[0]
print('Most common structure:', ensemble_df.loc[ensemble_df[plot_variable_1] == most_common_structure]['structure'].values[0])

# plot a bar chart of frequency of the top 5 structures in the path
plt.figure()
# get the top 5 structures
top_5 = ensemble_df[plot_variable_1].loc[sasa_min_path].value_counts().index[:5]
# plot the bar chart
plt.bar(ensemble_df.loc[ensemble_df[plot_variable_1].isin(top_5)]['structure'], ensemble_df[plot_variable_1].loc[sasa_min_path].value_counts().values[:5])
plt.xticks(rotation=90)
plt.ylabel('Frequency')
plt.show()

            

In [ ]:
# MC path finding by exchanging structures in a path of length N

use_ideal_initial_path = True

def calc_energy(path, path_length):

    wf_rmsd = 0
    wf_cv = 1

    # for minimising rmsd between structures on path
    energy_rmsd = 0
    for i in range(path_length - 1):
        energy_rmsd += (rmsd_matrix[np.where(ensemble_df.index.values == path[i])[0][0], np.where(ensemble_df.index.values == path[i+1])[0][0]]**2)
    energy_rmsd = wf_rmsd * np.sqrt ( 1/path_length * energy_rmsd )

    # for minimising pairwise differences in collective variable between structures on path
    energy_cv = 0
    for i in range(path_length -1):
        energy_cv += (cv_matrix[np.where(ensemble_df.index.values == path[i])[0][0], np.where(ensemble_df.index.values == path[i+1])[0][0]]**2)
    energy_cv = wf_cv * np.sqrt ( 1/path_length * energy_cv )

    # final term is sum
    energy = energy_rmsd + energy_cv

    # final energy
    return energy

# define a function that runs monte carlo simulated annealing to optimise path smoothness
def mc_path_optimisation(seed):

    # random seed
    np.random.seed(seed)

    mc_n_steps=10000
    initial_temperature = 0.00000

    # list of temperatures mapped to mc_n_steps increasing stepwise in 10 increments
    final_temperature = initial_temperature*1000
    temperatures = np.logspace(np.log10(initial_temperature), np.log10(final_temperature), num=mc_n_steps)

    mc_path_length = mc_n_bins

    mcpath = []

    # end states are the reference structures (get the index of the reference structures in the dataframe)
    first_structure = ensemble_df.sort_values(by=['rmsd_to_outward']).head(1).index.values[0]
    last_structure = ensemble_df.sort_values(by=['rmsd_to_inward']).head(1).index.values[0]

    #first_structure = #'ref_outward.pdb'
    #last_structure = #'ref_inward.pdb'

    #first_structure = ensemble_df.index.values[np.where(ensemble_df['structure'] == first_structure)[0][0]]
    #last_structure = ensemble_df.index.values[np.where(ensemble_df['structure'] == last_structure)[0][0]]

    # structure pool consists of the names of the structures in the ensemble dataframe
    structure_pool = ensemble_df.index.values

    # remove the first and last structures from the pool
    structure_pool = structure_pool[structure_pool != first_structure]
    structure_pool = structure_pool[structure_pool != last_structure]

    if use_ideal_initial_path == False:
        # randomly sample structures from each bin (excluding the first and last bins becasue they are the end states) for an initial guess path
        initial_guess_indices = []
        for i in range(mc_n_bins-2):
            initial_guess = ensemble_df.loc[ensemble_df['bin'] == i+1].sample(1)
            initial_guess_indices.append(initial_guess.index[0])
    else:
        initial_guess_indices = []
        initial_guess_indices = ideal_initial_path[1:-1]
        
    # add the end states to the initial guess path
    mcpath = np.concatenate((np.array([first_structure]), initial_guess_indices, np.array([last_structure])))

    # remove the randomly selected initial structures from the pool
    structure_pool = structure_pool[np.isin(structure_pool, mcpath[1:-1], invert=True)]

    # dictionary of paths and energies
    path_energies = {}

    # length of structure pool
    n_structure_pool = len(structure_pool)

    # run n_mc_runs steps
    for i in range(mc_n_steps):

        temperature = temperatures[i]

        # calculate energy of current path
        energy = calc_energy(mcpath, mc_path_length)
        
        # propose an exchange of a random (not endstate) structure in the path with a random structure from the pool
        new_mcpath = mcpath.copy()

        # select a random structure from the path that is not an endstate
        random_structure_in_path = np.random.choice(new_mcpath[1:-1])

        # select a random structure from the pool (and then remove it from the pool later if it is accepted)
        random_structure_in_pool = np.random.choice(structure_pool)
        # chose another random structure from the pool if the random structure is not in the ame bin as the structure it is replacing
        while ensemble_df.loc[random_structure_in_pool]['bin'] != ensemble_df.loc[random_structure_in_path]['bin']:
            random_structure_in_pool = np.random.choice(structure_pool)

        # replace the random structure in the new path with the random structure from the pool
        new_mcpath[np.where(new_mcpath == random_structure_in_path)[0][0]] = random_structure_in_pool

        # calcualte energy of proposed path
        new_energy = calc_energy(new_mcpath, mc_path_length)

        # calculate the difference between the two energies
        delta_energy = 0
        delta_energy = new_energy - energy

        # metropolis criterion
    
        # if lower, accept the new path
        if delta_energy < 0:
            #pass
            mcpath = new_mcpath

            # Insert the structure that was replaced into the middle of the structure pool (so the end states are always at the ends of the pool)
            structure_pool = np.insert(structure_pool, len(structure_pool) // 2, random_structure_in_path)
            # Remove the structure that was added to the path from the pool
            structure_pool = structure_pool[structure_pool != random_structure_in_pool]

            path_energies[new_energy] = mcpath
        
        else:
            #pass
            # if higher, accept with probability e^(-difference/temp)
            if np.random.rand() < np.exp(-delta_energy / temperature):

                mcpath = new_mcpath

                # Insert the structure that was replaced into the middle of the structure pool (so the end states are always at the ends of the pool)
                structure_pool = np.insert(structure_pool, len(structure_pool) // 2, random_structure_in_path)
                # Remove the structure that was added to the path from the pool
                structure_pool = structure_pool[structure_pool != random_structure_in_pool]

                path_energies[new_energy] = mcpath

            else:
                pass

        # check the structure pool is the correct length
        if len(structure_pool) != n_structure_pool:
            raise ValueError('Error: structure pool is the wrong length')
            
    # get the path with the lowest energy
    final_energy = min(path_energies.keys())
    final_path = path_energies[final_energy]
    relaxation_energies = list(path_energies.keys())
    final_path_structures = tuple(ensemble_df.loc[final_path]['structure'].values)

    return [final_energy, final_path, final_path_structures, relaxation_energies]

mc_n_runs = 10

# run mc_n_bins runs in parallel
with Pool(processes=num_processes) as pool:
    mc_runs = list(tqdm(pool.imap(mc_path_optimisation, range(mc_n_runs)), total=mc_n_runs))

# add the final energies and final path structures to a dataframe
mc_runs_df = pd.DataFrame(mc_runs, columns=['energy', 'path', 'path structures', 'relaxation energies'])
mc_runs_df = mc_runs_df.sort_values(by=['energy']) 

# display the best run
display(mc_runs_df[['energy', 'path structures']].head(3))

# plot relaxation of the best run
for i in range(1, 4):
    plt.plot(mc_runs_df.iloc[i]['relaxation energies'])
plt.show()

# calculate the energy for each point in the best path (mc_runs[1]) (for plotting)
energy_over_path = []
energy = {}
# get hte "path" of the best run (in indices)
test = mc_runs_df.iloc[1]['path']
for i in range(mc_n_bins - 1):
        energy[i] = (rmsd_matrix[np.where(ensemble_df.index.values == test[i])[0][0], np.where(ensemble_df.index.values == test[i+1])[0][0]]**2)
        energy[i] = np.sqrt ( 1 * energy[i] )
plt.plot(energy.values(), label='best')

test = mc_runs_df.iloc[-1]['path']
for i in range(mc_n_bins - 1):
        energy[i] = (rmsd_matrix[np.where(ensemble_df.index.values == test[i])[0][0], np.where(ensemble_df.index.values == test[i+1])[0][0]]**2)
        energy[i] = np.sqrt ( 1 * energy[i] )
plt.plot(energy.values(), label='worst')
# plot energy

# plot energy
plt.legend()
plt.show()

# write out the best run to a multistate pdb file and append timestamp to filename
with mda.Writer(base_directory + 'best_run.pdb', u.atoms.n_atoms) as W:
    # get the structures of the lowest energy path
    for structure in mc_runs_df.head(1)['path structures'].values[0]:
        # if structure is the reference, dont renumber residues
        if structure == 'ref_outward.pdb' or structure == 'ref_inward.pdb':
            # use a non-reference structure to make the universe
            u = mda.Universe(structure_directory + structure, 
                             structure_directory + structure) # make universe
            u.atoms.residues.resids += resid_offset 
            u.atoms.segments.segids = 'A'
            u.atoms.chainIDs = 'A'
            W.write(u.select_atoms('protein'))
        else:
            u = mda.Universe(structure_directory + structure, 
                             structure_directory + structure) # make universe
            u.atoms.residues.resids += resid_offset
            W.write(u.select_atoms('protein'))
        